# 0. Node Mapping - TRANSAKSI (With Live Progress)

Shows row-level progress so you can see it's working!

In [6]:
import os
import glob
import time
import lmdb
import shutil
import gc
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

# Configuration
ROOT_DIR = "/Volumes/Backup Plus/Zaman/graph"
DATA_DIR = os.path.join(ROOT_DIR, "data")
OUTPUT_LMDB_DIR = os.path.join(ROOT_DIR, "lmdb_node_mapping")

TRANSAKSI_DIR = os.path.join(DATA_DIR, "node_transaksi")
ID_COL = "id_trx"
MAP_SIZE = 1024 * 1024 * 1024 * 15  # 15GB

os.makedirs(OUTPUT_LMDB_DIR, exist_ok=True)

def format_time(seconds):
    if seconds < 60:
        return f"{seconds:.0f}s"
    elif seconds < 3600:
        return f"{seconds/60:.1f}m"
    else:
        return f"{seconds/3600:.1f}h"

def encode(x) -> bytes:
    return str(x).encode()

In [7]:
# Get files and estimate total rows
parquet_files = sorted(glob.glob(f"{TRANSAKSI_DIR}/*.parquet"))
print(f"Found {len(parquet_files)} parquet files\n")

# Count total rows first (fast metadata read)
print("Counting total rows (reading metadata only)...")
total_rows = 0
file_rows = []

for f in tqdm(parquet_files, desc="Scanning files"):
    pf = pq.ParquetFile(f)
    rows = pf.metadata.num_rows
    file_rows.append(rows)
    total_rows += rows

print(f"\nTotal rows to process: {total_rows:,}")
print(f"Files: {len(parquet_files)}")
print(f"\nRows per file:")
for f, r in zip(parquet_files, file_rows):
    print(f"  {os.path.basename(f)}: {r:,}")

Found 13 parquet files

Counting total rows (reading metadata only)...


Scanning files:   0%|          | 0/13 [00:00<?, ?it/s]


Total rows to process: 365,820,399
Files: 13

Rows per file:
  ds=202308.parquet: 24,502,104
  ds=202309.parquet: 24,685,551
  ds=202310.parquet: 24,993,720
  ds=202311.parquet: 27,709,892
  ds=202312.parquet: 28,092,720
  ds=202401.parquet: 25,036,696
  ds=202402.parquet: 27,478,727
  ds=202403.parquet: 29,351,474
  ds=202404.parquet: 29,240,609
  ds=202405.parquet: 30,884,592
  ds=202406.parquet: 30,901,863
  ds=202407.parquet: 30,566,853
  ds=202408.parquet: 32,375,598


In [8]:
def process_transaksi_with_progress():
    """Process with row-level progress bar."""
    print("="*60)
    print("Processing TRANSAKSI")
    print("="*60)
    
    start_time = time.time()
    
    # Prepare LMDB
    lmdb_path = os.path.join(OUTPUT_LMDB_DIR, "transaksi.lmdb")
    
    if os.path.exists(lmdb_path):
        shutil.rmtree(lmdb_path, ignore_errors=True)
        print(f"Removed existing database")
    
    env = lmdb.open(
        lmdb_path,
        map_size=MAP_SIZE,
        subdir=True,
        lock=True,
        readonly=False,
        max_dbs=1,
    )
    
    seen_hashes = set()
    counter = 0
    rows_processed = 0
    
    txn = env.begin(write=True)
    
    # Main progress bar for rows
    pbar = tqdm(total=total_rows, desc="Processing rows", unit="rows")
    
    for file_idx, pq_file in enumerate(parquet_files):
        file_name = os.path.basename(pq_file)
        pbar.set_postfix({"file": file_name, "unique": f"{counter:,}"})
        
        try:
            # Read in batches of 100k rows for better memory handling
            pf = pq.ParquetFile(pq_file)
            
            for batch in pf.iter_batches(batch_size=100_000, columns=[ID_COL]):
                ids = batch[ID_COL].to_pylist()
                batch_size = len(ids)
                
                for id_val in ids:
                    if id_val is None:
                        continue
                    
                    id_str = str(id_val)
                    id_hash = hash(id_str)
                    
                    if id_hash not in seen_hashes:
                        seen_hashes.add(id_hash)
                        txn.put(encode(id_str), encode(counter))
                        counter += 1
                
                rows_processed += batch_size
                pbar.update(batch_size)
                
                # Update stats
                elapsed = time.time() - start_time
                speed = rows_processed / elapsed if elapsed > 0 else 0
                eta = (total_rows - rows_processed) / speed if speed > 0 else 0
                pbar.set_postfix({
                    "file": f"{file_idx+1}/{len(parquet_files)}",
                    "unique": f"{counter:,}",
                    "speed": f"{speed:,.0f}/s",
                    "ETA": format_time(eta)
                })
                
                # Commit every 500k unique IDs
                if counter % 500_000 < 100:
                    txn.commit()
                    txn = env.begin(write=True)
                
                del ids
            
        except Exception as e:
            print(f"\nError processing {pq_file}: {e}")
            continue
        
        gc.collect()
    
    pbar.close()
    
    # Final commit
    txn.commit()
    env.close()
    
    elapsed = time.time() - start_time
    
    print(f"\n" + "="*60)
    print(f"✓ COMPLETED!")
    print(f"="*60)
    print(f"Total rows processed: {rows_processed:,}")
    print(f"Unique IDs mapped: {counter:,}")
    print(f"Duplicates skipped: {rows_processed - counter:,}")
    print(f"Time elapsed: {format_time(elapsed)}")
    print(f"Average speed: {rows_processed / elapsed:,.0f} rows/sec")
    print(f"Saved to: {lmdb_path}")
    
    del seen_hashes
    gc.collect()
    
    return counter

In [9]:
# RUN THIS CELL
transaksi_count = process_transaksi_with_progress()

Processing TRANSAKSI
Removed existing database


Processing rows:   0%|          | 0/365820399 [00:00<?, ?rows/s]

KeyboardInterrupt: 

In [ ]:
# Verify the output
lmdb_path = os.path.join(OUTPUT_LMDB_DIR, "transaksi.lmdb")
env = lmdb.open(lmdb_path, readonly=True, lock=False)

with env.begin() as txn:
    entries = txn.stat()['entries']
    print(f"Total entries in LMDB: {entries:,}")
    
    cursor = txn.cursor()
    print("\nSample entries:")
    for i, (key, value) in enumerate(cursor):
        if i >= 5:
            break
        print(f"  {key.decode()} -> {value.decode()}")

env.close()
print(f"\n✓ Transaksi mapping complete with {entries:,} unique nodes")